# File Integrity Monitor

A simple file integrity monitoring tool that uses SHA-256 hashing to detect modified, deleted, and new files.

## Imports

In [ ]:
from pathlib import Path
import hashlib
import json

## Select Folder

In [ ]:
folder = Path(input("Enter folder path: "))

## Create File Hashes

In [ ]:
file_hashes = {}

for file in folder.iterdir():
    if file.is_file():
        sha256_hash = hashlib.sha256()

        with open(file, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256_hash.update(chunk)

        file_hashes[file.name] = sha256_hash.hexdigest()

## Create or Load the Folder Baseline

In [ ]:
baselines_folder = Path("baselines")
baselines_folder.mkdir(exist_ok=True)

folder_id = hashlib.sha256(
    str(folder.resolve()).encode()
).hexdigest()

baseline_path = baselines_folder / f"{folder_id}.json"

if not baseline_path.exists():
    with open(baseline_path, "w") as f:
        json.dump(file_hashes, f, indent=4)

with open(baseline_path, "r") as f:
    saved_hashes = json.load(f)

## Calculate Current Hashes

In [ ]:
current_hashes = {}

for file in folder.iterdir():
    if file.is_file():
        sha256_hash = hashlib.sha256()

        with open(file, "rb") as f:
            for chunk in iter(lambda: f.read(4096), b""):
                sha256_hash.update(chunk)

        current_hashes[file.name] = sha256_hash.hexdigest()

## Compare Files

In [ ]:
modified_files = []
deleted_files = []
new_files = []

for file in current_hashes:
    if file not in saved_hashes:
        new_files.append(file)

    elif current_hashes[file] != saved_hashes[file]:
        modified_files.append(file)

for file in saved_hashes:
    if file not in current_hashes:
        deleted_files.append(file)

## Display Results

In [ ]:
print("\nModified Files:")
if modified_files:
    for file in modified_files:
        print(f"- {file}")
else:
    print("None")

print("\nDeleted Files:")
if deleted_files:
    for file in deleted_files:
        print(f"- {file}")
else:
    print("None")

print("\nNew Files:")
if new_files:
    for file in new_files:
        print(f"- {file}")
else:
    print("None")